In [151]:
from selenium import webdriver as wd
from dotenv import load_dotenv
import os
from selenium.webdriver.common.by import By

In [152]:
def get_target_sido_list(default_val=['서울']):
    """
    .env 파일에서 TARGET_SIDO_LIST를 읽어와 리스트로 반환하는 함수
    """
    raw_target_sido = os.getenv('TARGET_SIDO_LIST')
    
    if not raw_target_sido:
        return default_val
    
    return [name.strip() for name in raw_target_sido.split(',')]

In [153]:
# dotenv로 환경변수 불러오기
load_dotenv()
STARBUCKS_URL        = os.getenv('STARBUCKS_URL')
WAIT_TIME            = os.getenv('WAIT_TIME')
REGION_SEARCH_BUTTON = os.getenv('REGION_SEARCH_BUTTON')
SLEEP_TIME           = os.getenv('SLEEP_TIME')
TARGET_SIDO_LIST     = get_target_sido_list()
#print(TARGET_SIDO_LIST)

In [154]:
driver = wd.Chrome()

In [155]:
driver.get(STARBUCKS_URL)
# 안정성을 위해 화면 로드될 때까지 대기
driver.implicitly_wait(WAIT_TIME)

In [156]:
# 화면에서 지역검색 클릭 후 대기
region_tab = driver.find_element(By.LINK_TEXT, REGION_SEARCH_BUTTON)
driver.execute_script("arguments[0].click();", region_tab) # 커피콩 대기 화면 무시 후 클릭하기

#import time
#time.sleep(int(SLEEP_TIME))

In [157]:
# CSS_SELECTOR 
sidos = driver.find_elements(By.CSS_SELECTOR, "a.set_sido_cd_btn")
#len(sidos)

In [158]:
sidos_value = [
    {
        'code' : sido.get_attribute("data-sidocd").strip(),
        'name' : sido.text
    } 
    for sido in sidos
]

print(sidos_value)

[{'code': '01', 'name': '서울'}, {'code': '08', 'name': '경기'}, {'code': '02', 'name': '광주'}, {'code': '03', 'name': '대구'}, {'code': '04', 'name': '대전'}, {'code': '05', 'name': '부산'}, {'code': '06', 'name': '울산'}, {'code': '07', 'name': '인천'}, {'code': '09', 'name': '강원'}, {'code': '10', 'name': '경남'}, {'code': '11', 'name': '경북'}, {'code': '12', 'name': '전남'}, {'code': '13', 'name': '전북'}, {'code': '14', 'name': '충남'}, {'code': '15', 'name': '충북'}, {'code': '16', 'name': '제주'}, {'code': '17', 'name': '세종'}]


In [159]:
import pandas as pd
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 최종 데이터를 통합해서 저장할 리스트 (반복문 밖에서 선언)
total_stores_data = []

for sido_v in sidos_value:
    if sido_v['name'] in TARGET_SIDO_LIST: 
        # 1. 해당 지역(서울 등) 버튼 찾아서 클릭
        sido_selector = f"a.set_sido_cd_btn[data-sidocd='{sido_v['code']}']"
        sido_btn = driver.find_element(By.CSS_SELECTOR, sido_selector)
        driver.execute_script("arguments[0].click();", sido_btn)
        time.sleep(5)
        
        # 2. 구군 '전체' 버튼 찾아서 클릭 (데이터 로딩 핵심!)
        all_btn = driver.find_element(By.CSS_SELECTOR, "a.set_gugun_cd_btn[data-guguncd='']")
        driver.execute_script("arguments[0].click();", all_btn)
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "li.quickResultLi"))
            )
            time.sleep(2) # 리스트가 다 그려질 수 있도록 추가 여유 시간
        except:
            print("⏳ 로딩 시간이 너무 길어 요소를 찾지 못했습니다.") # 리스트가 나타날 때까지 넉넉히 대기
        
        # 4. 해당 지역의 모든 매장 정보(li 태그) 가져오기
        stores = driver.find_elements(By.CSS_SELECTOR, "li[data-name]")
        print(f"[LOG] 찾은 매장 개수: {len(stores)}개")

        for store in stores:
            # 매장명
            name = store.get_attribute("data-name")
            # [수정된 주소 추출 로직]
            try:
                # .text 대신 innerText를 쓰면 줄바꿈이 포함된 텍스트를 더 잘 가져와
                full_info = store.find_element(By.CSS_SELECTOR, "p.result_details").get_attribute("innerText")
                
                # 주소는 보통 첫 번째 줄에 있고, 전화번호(1522-3232) 앞에 있어
                # 줄바꿈(\n)을 기준으로 나누어 첫 번째 요소만 가져오면 주소야!
                address = full_info.split('\n')[0].strip() 
                
                # 만약 split('\n')으로도 안 된다면 전화번호 기준으로 다시 한 번 자르기
                if "1522-3232" in address:
                    address = address.split("1522-3232")[0].strip()
                    
            except:
                address = "주소 확인 불가"
            # 매장 타입 (클래스명 추출)
            icon_class = store.find_element(By.TAG_NAME, "i").get_attribute("class")
            
            # 타입 분류 로직
            if "pin_generalDT" in icon_class:
                s_type = "generalDT"
            elif "pin_reserve" in icon_class:
                s_type = "reserve"
            else:
                s_type = "general"
                
            # 리스트에 딕셔너리 형태로 추가
            total_stores_data.append({
                "매장명": name,
                "주소": address,
                "매장타입": s_type,
                "지역": sido_v['name'] # 확장성을 위해 지역 정보도 저장
            })

⏳ 로딩 시간이 너무 길어 요소를 찾지 못했습니다.
[LOG] 찾은 매장 개수: 705개


In [160]:
# 5. CSV 저장 (모든 반복이 끝난 후)
if total_stores_data:
    df = pd.DataFrame(total_stores_data)
    
    # 파일명에 수집 지역들을 포함하거나 날짜를 넣으면 더 좋겠지?
    filename = f"starbucks_{'_'.join(TARGET_SIDO_LIST)}.csv"
    
    # utf-8-sig로 저장해야 엑셀에서 한글이 안 깨져!
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {filename} (총 {len(total_stores_data)}개 매장)")
else:
    print("❌ 수집된 데이터가 없습니다.")

✅ 저장 완료: starbucks_서울.csv (총 705개 매장)


In [161]:
driver.quit()